In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft, rfft, rfftfreq
from scipy.signal.windows import gaussian
from scipy.signal import convolve


def fftcrosscorr(x,y,dlen=10000):
    
    # make sure that we have odd number of signals as it makes fft easier
    if (dlen%2 == 0):
        dlen -= 1
        
    # the fft coecofficents of the crosscorrelation function c_xy(t)
    dt = x[1,0] - x[0,0] # assume the timestep is constant
    window = len(x)//dlen
    omega0 = 2.0*np.pi/(dlen-1)/dt
    cxyomega = np.zeros((dlen,2),dtype=np.complex_)
    cxyomega[0:dlen//2 + 1,0] = np.arange(dlen//2 + 1)*omega0
    cxyomega[dlen//2 + 1:,0] = np.arange(dlen//2,0,-1)*omega0*-1
    
    for i in range(window):
        dx = x[i*dlen:(i+1)*dlen,1] 
        dy = y[i*dlen:(i+1)*dlen,1]
        Ax = np.fft.fft(dx[:] , axis = 0)
        Ay = np.fft.fft(dy[:] , axis = 0)

        cxyomega[:,1] += np.conjugate(Ax[:]) * Ay[:]/dlen*dt
        
    for i in range(window-1):
        dx = x[i*dlen+dlen//2:(i+1)*dlen+dlen//2,1] 
        dy = y[i*dlen+dlen//2:(i+1)*dlen+dlen//2,1]
        Ax = np.fft.fft(dx[:] , axis = 0)
        Ay = np.fft.fft(dy[:] , axis = 0)

        cxyomega[:,1] += np.conjugate(Ax[:]) * Ay[:]/dlen*dt
        
    cxyomega[:,1]/=(window*2-1)
    return cxyomega

def smooth_signal(signal, window_size=51, sigma=7):
    # Gaussian kernel for convolution and normalization
    kernel = gaussian(window_size, std=sigma)
    kernel /= np.sum(kernel)
    smooth_signal = convolve(signal, kernel, mode='same')
    return smooth_signal

def normarlize_area(omega, intensity):
    area = np.trapz(intensity, omega)
    return intensity/area

In [2]:
def Raman_plot_tensor(total_alpha, dt=0.25, dlen=5000, length=400, window_size=20, sigma=3, temperature=300.0):
    """
    total_alpha shape: (N_frames, 9) (xx, xy, xz, yx, yy, yz, zx, zy, zz)
    """
    N_frames = len(total_alpha)
    time_arr = np.arange(N_frames) * dt
    
    # 1. Isotropic (Trace / 3)
    a_t = (total_alpha[:, 0] + total_alpha[:, 4] + total_alpha[:, 8]) / 3.0
    delta_a = a_t - np.mean(a_t)
    
    a_input = np.zeros((N_frames, 2))
    a_input[:, 0] = time_arr
    a_input[:, 1] = delta_a
    
    # Isotropic Power Spectrum
    ft_iso = fftcrosscorr(a_input, a_input, dlen=dlen)
    power_iso = ft_iso[:, 1].real
    
    # 2. Anisotropic
    beta = np.copy(total_alpha)
    beta[:, 0] -= a_t  # xx - a
    beta[:, 4] -= a_t  # yy - a
    beta[:, 8] -= a_t  # zz - a
    
    power_aniso = np.zeros_like(power_iso)
    # Tr(beta*beta)
    for k in range(9):
        delta_beta_k = beta[:, k] - np.mean(beta[:, k])
        beta_input = np.zeros((N_frames, 2))
        beta_input[:, 0] = time_arr
        beta_input[:, 1] = delta_beta_k
        
        ft_beta_k = fftcrosscorr(beta_input, beta_input, dlen=dlen)
        power_aniso += ft_beta_k[:, 1].real
    
    # 3. Wavenumber
    omega = ft_iso[:, 0] * 1e15 / 2.99792458e10 / (2 * np.pi)

    intensity_iso = omega **2 * power_iso  
    intensity_aniso = omega **2 * power_aniso 

    smooth_iso = smooth_signal(intensity_iso, window_size=window_size, sigma=sigma)
    smooth_aniso = smooth_signal(intensity_aniso, window_size=window_size, sigma=sigma)
    
    return omega[:length], smooth_iso[:length], smooth_aniso[:length]

def load_md_raman(pkl_path, temp=300.0, dlen=5000, length=600, window_size=30, sigma=1.5):
    filename ='total_alpha_epsilon.pkl'
    pkl = f'{pkl_path}/{filename}'
    with open(pkl, 'rb') as f:
        data_dict = pickle.load(f)
    total_alpha_tensor = data_dict['total_alpha'] 
    
    return Raman_plot_tensor(
        total_alpha_tensor, dt=0.25, dlen=dlen, length=length, 
        window_size=window_size, sigma=sigma, temperature=temp
    )

# sim example
md_v, md_iso, md_aniso = load_md_raman(path, temp=temp_val, dlen=dlen, length=length, window_size=window_s, sigma=sigma)
norm_iso = normalize_area(md_v, md_iso)
norm_aniso = normalize_area(md_v, md_aniso)